# Suite TYP — Types

`Types` renders schemas as Python dataclasses and reads dataclasses back as schemas, step by step: a session runs over
the schemas a store registers, Python's syntax trees and the output, which holds the module. `Dataclass` takes one
decision per object schema, whether its class is frozen; `Schema` reads one class back. Both round trips are tested:
schemas to source to schemas, and source to schemas to source.

In [ ]:
import { Types } from "@mbse/codegen-python";
import { Transforms as T } from "@mbse/patterns";
import { Python312, Syntax as Py } from "@mbse/programs/Python";
import { JSON, Modules, Proxies, Schemas as S, Validators as V } from "@mbse/schemas/Framework";
import { ValueError } from "@mbse/schemas/Framework/Errors";
import { assert, equal, native, raises, same, stocked } from "./support.js";

const B = Py.LANGUAGE.Builders as any;
const Phone = new S.OfObject.Builder().name("Phone").properties(native("number")).create();
const Contact = new S.OfObject.Builder().name("Contact").ref().description('A "person"\nwith a home').properties(
  native("name"), native("age", BigInt), (p) => p.name("home").of(Phone),
  (p) => p.name("tags").of((t) => t.as_indexed((i) => i.of((x) => x.as_native(String)))),
  (p) => p.name("phones").of((t) => t.as_indexed((i) => i.of(Phone)))).create();
const Empty = new S.OfObject.Builder().name("Empty").create();

/** The schemas as a module's JSON text, to compare schemas by what they say. */
function module_of(schemas: unknown[]): string {
  const store = new Proxies.OfStore();
  return JSON.ToJSON(store)(S.Module.Schema, Modules.module(store, schemas as never) as never);
}

/** The schemas a reading session's store registers, by name. */
function registered(session: T.Session, ...names: string[]): any[] {
  const schemas = (session.store as any).stores[0].store;
  return names.map((name) => schemas.registered(name));
}

const SOURCE = `from __future__ import annotations
from dataclasses import dataclass


@dataclass(eq=False)
class Contact:
    "A \\"person\\"\\nwith a home"
    name: str | None = None
    age: int | None = None
    home: Phone | None = None
    tags: list[str] | None = None
    phones: list[Phone] | None = None


@dataclass
class Empty:
    pass


@dataclass(frozen=True)
class Phone:
    number: str | None = None
`;

## TYP-01 · The store: the schemas a store registers and refers to, Python's syntax trees, and the output holding the module

In [ ]:
{
  const module = B.Module().create();
  const store = Types.store(stocked(Contact, Empty), module);
  assert(same(store.extent("Schemas.Object"), [Contact, Empty, Phone])); // Phone: referred to
  const output = store.singleton(Types.OUTPUT) as unknown as Types.Output;
  assert(output.module === module && output.schema_name() === "Codegen.Output" && output.owner() === null);
  assert(output.identity() === output.identity() && output.identity() !== new Types.Output().identity());
  assert(same((V.entries_of(output as never).get("modules") ?? []).map((e) => e.targets.get("module")), [module])
    && V.entries_of(new Types.Output() as never).size === 0);
  assert(Types.Output.Schema.singleton === "Codegen.Output" && same(Types.Generated.links, ["output", "module"]));
  assert(same(store.extent("Programs.Python.Module"), [module])); // reached from the output
}

## TYP-02 · Dataclass: a candidate per value of `frozen` for each object schema it renders and has no class for

In [ ]:
{
  const session = new T.Session(Types.store(stocked(Contact, Empty), B.Module().create()), [...Types.TO_PYTHON]);
  assert(equal(session.candidates().map((c) => [(c.match["s"] as any).name, c.arguments]), [
    ["Contact", { frozen: false }], ["Contact", { frozen: true }], ["Empty", { frozen: false }],
    ["Empty", { frozen: true }], ["Phone", { frozen: false }], ["Phone", { frozen: true }]]));
  assert(Types.Dataclass.parameters.get("frozen")?.description === "Whether the class is frozen");
  const Link = new S.OfRelation.Builder().name("Link").links("a", "b").create();
  const unrendered = [
    new S.OfObject.Builder().name("Inline").properties((p) => p.name("x").of((t) => t.as_object((o) => o))).create(),
    new S.OfObject.Builder().name("Keyed").properties((p) => p.name("x").of(
      (t) => t.as_indexed((i) => i.key((k) => k.as_native(String)).of(Phone)))).create(),
    new S.OfObject.Builder().name("Foreign").properties((p) => p.name("x").of(
      (t) => t.as_native((n: any) => n.token("ccpp", "int32_t")))).create(),
    new S.OfObject.Builder().name("Nested").properties((p) => p.name("x").of(
      (t) => t.as_indexed((i) => i.of((x) => x.as_indexed((j) => j.of(Phone)))))).create(),
    new S.OfObject.Builder().name("Generic").parameters((q) => q.name("n")).create(),
    new S.OfObject.Builder().name("Linked").relations((r) => r.name("links").of(Link).me("a")).create()];
  const others = new T.Session(Types.store(stocked(...unrendered), B.Module().create()), [...Types.TO_PYTHON]);
  assert(same([...new Set(others.candidates().map((c) => (c.match["s"] as any).name))], ["Phone"])); // referred to by Keyed; the others render no class
}

## TYP-03 · Taking the steps: the caller's decisions and the policy's, the module printed, the trace

In [ ]:
{
  const session = new T.Session(Types.store(stocked(Contact, Empty), B.Module().create()), [...Types.TO_PYTHON]);
  const phone = session.candidates().find((c) => c.match["s"] === Phone && c.arguments["frozen"] === true) as T.Candidate;
  session.take(phone); // the caller decides that Phone is frozen
  assert(session.run(Types.PLAIN) === 2 && session.done);
  assert(Types.text(session) === SOURCE);
  assert(equal(session.steps.map((step) => [step.transform, step.match, step.arguments, step.by]), [
    ["Dataclass", [["s", "Schemas.Object#2"]], [["frozen", true]], "caller"],
    ["Dataclass", [["s", "Schemas.Object#0"]], [["frozen", false]], "policy"],
    ["Dataclass", [["s", "Schemas.Object#1"]], [["frozen", false]], "policy"]]));
  assert(Types.text(Types.generate(stocked(Contact, Empty))) === SOURCE.replace("@dataclass(frozen=True)", "@dataclass"));
  const frozen = Types.generate(stocked(Contact), new T.Policy(new T.Clause("Dataclass", { frozen: true })));
  assert(Types.text(frozen).includes("@dataclass(eq=False, frozen=True)\nclass Contact:"));
  assert(Py.LANGUAGE.validate((session.store.singleton(Types.OUTPUT) as unknown as Types.Output).module as never).length === 0); // a valid tree
}

## TYP-04 · Schema: a dataclass read back as a schema; names, references, lists, docstrings, `eq=False`; refusals

In [ ]:
{
  const reading = Types.read(Python312.parse(SOURCE) as never);
  const [contact, empty, phone] = registered(reading, "Contact", "Empty", "Phone");
  assert(contact.ref === true && contact.description === 'A "person"\nwith a home'
    && same([...contact.properties.keys()], ["name", "age", "home", "tags", "phones"]));
  assert(contact.properties.get("home").type === phone && contact.properties.get("phones").type.item === phone); // filled when read
  assert(contact.properties.get("age").type.equals(new S.OfNative.Data(BigInt))
    && contact.properties.get("tags").type.item.equals(new S.OfNative.Data(String)));
  assert(empty.properties.size === 0 && phone.ref === false && same([...phone.properties.keys()], ["number"]));
  assert(same(reading.steps.map((step) => step.transform), ["Schema", "Schema", "Schema"]) && reading.done);
  const written = `@dataclass
class Plain:
    """A 'quoted' \\\\ text"""
    count: int
    note: str = "x"


class Ignored:
    x: int


@dataclass(frozen=True, eq=True)
class Frozen:
    f()
    x: int


@other
class Other:
    x: int
`;
  const target = stocked(Phone);
  Types.read(Python312.parse(written) as never, target);
  assert((target.registered("Plain") as any).description === "A 'quoted' \\ text" && !(target.registered("Frozen") as any).ref);
  assert(same([...(target.registered("Plain") as any).properties.keys()], ["count", "note"]) && !target.names().includes("Ignored"));
  assert((target.registered("Frozen") as any).description === null && !target.names().includes("Other"));
  assert(target.names().includes("Phone")); // a store given is the one read into
  assert(Types.read(Python312.parse("@dataclass\nclass Single:\n    'single'\n") as never).done);
  assert(registered(Types.read(Python312.parse("@dataclass\nclass Single:\n    'it\\'s'\n") as never), "Single")[0].description === "it's");
  for (const annotation of ["dict[str, int]", "typing.Any", "x.y[int]", "int | str", "int | 3", "int + None"]) {
    raises(ValueError, () => Types.read(Python312.parse(`@dataclass\nclass Bad:\n    x: ${annotation}\n`) as never),
      `Bad.x: cannot read the annotation ${annotation}`);
  }
}

## TYP-05 · Round trips: schemas to source to the same schemas, and source to schemas to the same source

In [ ]:
{
  const back = registered(Types.read(Python312.parse(SOURCE) as never), "Contact", "Empty", "Phone");
  assert(module_of(back) === module_of([Contact, Empty, Phone])); // what the schemas say, frozen aside
  const session = new T.Session(Types.store(stocked(...back), B.Module().create()), [...Types.TO_PYTHON]);
  session.take(session.candidates().find((c) => c.match["s"] === back[2] && c.arguments["frozen"] === true) as T.Candidate);
  session.run(Types.PLAIN);
  assert(Types.text(session) === SOURCE); // the decision the trace keeps, taken again
  assert(Types.text(Types.read(Python312.parse(SOURCE) as never)) === Python312.print(Python312.parse(SOURCE))); // read leaves the module
}